# 🛡️ Policy Enforcement 101
**Agent Governance Toolkit — Interactive Demo**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/agent-governance-toolkit/blob/main/notebooks/01_policy_enforcement_101.ipynb)

In this notebook you will:
- Define guardrails as an ACS manifest plus a Rego bundle
- Evaluate actions against an ACS policy manifest
- See violations get blocked in real time
- Inspect the audit trail

> **No API key required** — this demo runs fully offline.

## Step 1 — Install the toolkit

In [ ]:
!pip install agent-governance-toolkit[full] -q

## Step 2 — Define a Governance Policy

In [ ]:
from pathlib import Path

import yaml

# An ACS policy is a manifest plus a Rego bundle. Write both, then load them.
policy_dir = Path("policies")
policy_dir.mkdir(exist_ok=True)

manifest = {
    "agent_control_specification_version": "0.3.1-beta",
    "metadata": {"name": "policy-enforcement-101", "version": "1.0"},
    "extends": [],
    "policies": {
        "guardrails": {
            "type": "rego",
            "bundle": ".",
            "query": "data.agt.notebooks.policy101.result",
        }
    },
    "intervention_points": {
        "input": {"policy_target": "$.input.body", "policy": {"id": "guardrails"}}
    },
}
(policy_dir / "manifest.yaml").write_text(yaml.safe_dump(manifest, sort_keys=False))

(policy_dir / "guardrails.rego").write_text("""
package agt.notebooks.policy101

import rego.v1

text := sprintf("%v", [input.policy_target.value])

blocked_patterns := {
    "DROP TABLE": `(?i)drop\\s+table`,
    "rm -rf": `(?i)rm\\s+-rf`,
    "SSN": `\\b\\d{3}-\\d{2}-\\d{4}\\b`,
}

hit contains name if {
    some name, pattern in blocked_patterns
    regex.match(pattern, text)
}

budgets := object.get(object.get(object.get(input, "snapshot", {}), "envelope", {}), "budgets", {})

budget_exceeded if object.get(budgets, "tool_call_count", 0) >= 5

result := {"decision": "deny", "reason": "budget_tool_calls_exceeded"} if budget_exceeded

result := {"decision": "deny", "reason": sprintf("blocked_pattern:%v", [concat(",", sort(hit))])} if {
    not budget_exceeded
    count(hit) > 0
}

result := {"decision": "allow", "reason": "no_rule_matched"} if {
    not budget_exceeded
    count(hit) == 0
}
""".lstrip())

print("Policy written to", policy_dir / "manifest.yaml")
print("Blocked patterns: DROP TABLE, rm -rf, SSN")
print("Tool-call budget: 5")

## Step 3 — Create a LangChain Governed Agent

In [ ]:
from agent_control_specification import AgentControl
from agent_os.integrations import LangChainKernel

runtime = AgentControl("policies/manifest.yaml")
kernel = LangChainKernel(runtime=runtime)
ctx = kernel.create_context("demo-agent")
audit = []

print("Runtime, kernel and context created successfully.")

## Step 4 — Test Policy Violations

In [ ]:
from datetime import datetime

test_inputs = [
    ("DROP TABLE users; SELECT 1",  "Dangerous SQL"),
    ("Run: rm -rf /var/logs",        "Destructive shell command"),
    ("My SSN is 123-45-6789",        "PII — SSN pattern"),
    ("What is the weather in London?", "Safe query"),
]

print(f"{'Input':<45} {'Result':<10} Reason")
print("-" * 80)

for text, label in test_inputs:
    allowed, reason = kernel.pre_execute(ctx, text)
    status = "✅ ALLOWED" if allowed else "🚫 BLOCKED"
    print(f"{label:<45} {status:<10} {reason}")
    audit.append({
        "ts": datetime.now().isoformat(),
        "label": label,
        "status": "ALLOWED" if allowed else "BLOCKED",
        "reason": reason,
    })

## Step 5 — Test Call Budget Enforcement

In [ ]:
print("Simulating call budget exhaustion...")
ctx.call_count = 5

allowed, reason = kernel.pre_execute(ctx, "Summarise the quarterly report")
print(f"Status: {'✅ ALLOWED' if allowed else '🚫 BLOCKED'}")
print(f"Reason: {reason}")

ctx.call_count = 0  # reset

## Step 6 — View Audit Trail

In [ ]:
print("\n── Audit Trail ──────────────────────────────────────")
for i, entry in enumerate(audit, 1):
    print(f"  [{i}] {entry['ts']}")
    print(f"       Input:  {entry['label']}")
    print(f"       Status: {entry['status']}")
    print(f"       Reason: {entry['reason']}")
    print()

blocked = sum(1 for e in audit if e['status'] == 'BLOCKED')
allowed = len(audit) - blocked
print(f"Summary: {allowed} allowed, {blocked} blocked out of {len(audit)} total")

## ✅ What You Learned

- How to define an ACS policy with blocked patterns and call budgets
- How the governance layer intercepts agent actions before execution
- How to inspect the audit trail for compliance reporting

**Next:** Try the [MCP Security Proxy notebook →](./02_mcp_security_proxy.ipynb)